In [34]:
# Set seed for reproducibility
SEED = 45

# Import necessary libraries
import os

# Set environment variables before importing modules
# Set PYTHONHASHSEED for deterministic hash values
os.environ['PYTHONHASHSEED'] = str(SEED) 
# Set MPLCONFIGDIR to avoid creating config files in the home directory
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/' 

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python's 'random'
np.random.seed(SEED)
random.seed(SEED)

# --- PyTorch Setup and Device Configuration ---
import torch
torch.manual_seed(SEED)
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader

logs_dir = "tensorboard"

# 1. Check for CUDA (NVIDIA GPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    # Enable cuDNN benchmark for faster, but sometimes less reproducible, training
    # Set to False if absolute reproducibility is paramount
    torch.backends.cudnn.benchmark = True 
# 3. Default to CPU
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
# ---------------------------------------------

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings (No more %matplotlib inline)
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)

PyTorch version: 2.6.0+cu124
Device: cuda


## 🔄 **Data Preprocessing**

In [35]:
# Load the dataset from a CSV file
df_train = pd.read_csv("/kaggle/input/pirate_pain_train.csv")
df_public_test = pd.read_csv("/kaggle/input/pirate_pain_test.csv")
df_labels = pd.read_csv("/kaggle/input/pirate_pain_train_labels.csv")

In [36]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_labels['label_encoded'] = label_encoder.fit_transform(df_labels['label'])
# This gives: ['high_pain', 'low_pain', 'no_pain']
label_map = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2}
df_labels['label_encoded'] = df_labels['label'].map(label_map)
df_labels.drop(["label"], axis =1)

,sample_index,label_encoded
0,0,0
1,1,0
2,2,1
3,3,0
4,4,0
...,...,...
656,656,0
657,657,0
658,658,0
659,659,0


In [37]:
from sklearn.preprocessing import MinMaxScaler
def preProcess_with_2D_PE(df: pd.DataFrame, scaler: MinMaxScaler, fit_scaler: bool = True):
    
    # Define the period (max step index + 1)
    PERIOD = 160
    
    # --- 1. Feature Engineering and Dropping Columns (Same as before) ---
    # ... (Your pain_survey, merged_n_features, and column drops)
    pain_cols = ["pain_survey_1", "pain_survey_2", "pain_survey_3", "pain_survey_4"]
    df["pain_survey"] = np.floor(df[pain_cols].median(axis=1)).astype(int)
    df.drop(columns=pain_cols, inplace=True)
    
    df['merged_n_features'] = np.where(
        (df['n_legs'] == 'two') & (df['n_hands'] == 'two') & (df['n_eyes'] == 'two'),
        0, 
        1 
    )
    df.drop(columns=['n_legs', 'n_hands', 'n_eyes', "joint_30"], inplace=True)
    
    # --- 2. 2D Sinusoidal Encoding (Replacing 'time') ---
    
    # Sinusoidal formula: sin(2 * pi * t / P) and cos(2 * pi * t / P)
    df["time_sin"] = np.sin(2 * np.pi * df["time"] / PERIOD)
    df["time_cos"] = np.cos(2 * np.pi * df["time"] / PERIOD)
    
    # Drop the original linear 'time' column
    df.drop(columns=["time"], inplace=True)
    
    # --- 3. Scaling the Non-PE Features ---
    
    # Columns to scale are all columns EXCEPT 'sample_index'
    # The new 'time_sin' and 'time_cos' columns are already normalized (-1 to 1)
    # They should generally be excluded from the StandardScaler
    pe_cols = ["time_sin", "time_cos"]
    cols_to_scale = [col for col in df.columns if col not in ['sample_index'] + pe_cols]
    
    if fit_scaler:
        df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
    else:
        df[cols_to_scale] = scaler.transform(df[cols_to_scale])

feature_scaler = MinMaxScaler()
preProcess_with_2D_PE(df_train, feature_scaler, fit_scaler=True)
preProcess_with_2D_PE(df_public_test, feature_scaler, fit_scaler=False)

In [38]:
from sklearn.model_selection import train_test_split

user_labels = df_labels.copy()

train_users, val_users = train_test_split(
    user_labels['sample_index'],
    test_size=0.2,  # 20% validation
    stratify=user_labels['label'], # This is the key for imbalance
    random_state=SEED
)

df_train_full = df_train.copy()

# Filter the full dataframe to get rows for train/val users
df_train_fold = df_train_full[df_train_full['sample_index'].isin(train_users)]
df_val_fold   = df_train_full[df_train_full['sample_index'].isin(val_users)]

# Merge labels back in
df_train_fold = df_train_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')
df_val_fold   = df_val_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')

df_train_fold = df_train_fold.drop(columns=['label'])
df_val_fold = df_val_fold.drop(columns=['label'])

In [39]:
## Casting
df_train_fold = df_train_fold.astype('float32')
df_val_fold   = df_val_fold.astype('float32')

df_train_fold['label_encoded']  = df_train_fold['label_encoded'].astype('int64')
df_val_fold['label_encoded']   = df_val_fold['label_encoded'].astype('int64')

In [40]:
feature_cols = [col for col in df_train_fold.columns if 'joint_' in col]
feature_cols.extend(['pain_survey', 'merged_n_features'])


X_data_2d = df_train_fold[[col for col in df_train_fold.columns if col.startswith('joint_') or col in ['pain_survey', 'merged_n_features']]].values
Y_target_1d = df_train_fold['label_encoded'].values

# Check final count of X features:
num_features = X_data_2d.shape[1]

print(f"Raw X shape: {X_data_2d.shape}")
print(f"Raw Y shape: {Y_target_1d.shape}")

Raw X shape: (84480, 32)
Raw Y shape: (84480,)


### IMPORTANT
Now the CNN it's built, the CNN takes in input the time series with the slicing windows.
For this reason i guess that is possible to proceede as always and after that continue with the training.

## Prepare data for training

In [41]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [42]:
def build_sequences(df, feature_cols, id_col='sample_index', label_col='label_encoded', window=200, stride=200):
    """
    Builds sequences from a time-series dataframe.
    
    Args:
        df (pd.DataFrame): The input DataFrame (e.g., df_train_fold) containing
                           features, IDs, and labels.
        feature_cols (list): A list of column names to be used as features.
        id_col (str): The name of the column for unique sample IDs.
        label_col (str): The name of the column for the labels.
        window (int): The size of each sequence (window).
        stride (int): The step size between sequences.
    """
    # Sanity check
    # assert window % stride == 0
    
    num_features = len(feature_cols)
    dataset = []
    labels = []

    # Iterate over unique sample IDs
    for sample_id in df[id_col].unique():
        
        # Get the dataframe for the current sample
        temp_df = df[df[id_col] == sample_id]

        # Extract feature data for the current ID
        temp_features = temp_df[feature_cols].values

        # Retrieve the single label for the current ID
        # (Assumes all rows for one ID have the same label)
        label = temp_df[label_col].values[0]

        # Calculate padding length to ensval_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drure full windows
        # This logic correctly handles cases where length is already a multiple
        padding_len = (window - len(temp_features) % window) % window
        
        if padding_len > 0:
            # Create zero padding and concatenate with the data
            padding = np.zeros((padding_len, num_features), dtype='float32')
            temp_features = np.concatenate((temp_features, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp_features):
            dataset.append(temp_features[idx:idx + window])
            labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    labels = np.array(labels)

    return dataset, labels

## 🛠️ **Model Building**

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.
    Reference: https://arxiv.org/abs/1708.02002 (Lin et al. 2017)

    Args:
        alpha (float or list): Weighting factor for classes (balances class imbalance)
        gamma (float): Focusing parameter to reduce the loss for well-classified examples
        reduction (str): 'none' | 'mean' | 'sum'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        if isinstance(alpha, (float, int)):
            self.alpha = torch.tensor([alpha])
        else:
            self.alpha = torch.tensor(alpha) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs: Predictions (logits), shape [batch_size, num_classes]
            targets: Ground truth labels, shape [batch_size]
        """
        # Compute log-probabilities
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)

        # Select log-probability of the correct class
        log_probs_true = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        probs_true = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        # Compute focal weight
        focal_weight = (1 - probs_true) ** self.gamma

        # Apply alpha (class weight)
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_factor = self.alpha[targets]
            focal_weight = alpha_factor * focal_weight

        # Compute final loss
        loss = -focal_weight * log_probs_true

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


In [44]:
import math
import torch
from torch.optim import Optimizer

class Ranger(Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.5, k=6, betas=(0.95, 0.999), eps=1e-5, weight_decay=0):
        """
        Ranger = RAdam + Lookahead
        Args:
            params: model parameters
            lr: learning rate
            alpha: lookahead step size (0.5 is default)
            k: lookahead steps before sync (6 is default)
            betas: RAdam betas
            eps: numerical stability
            weight_decay: L2 regularization
        """
        defaults = dict(lr=lr, alpha=alpha, k=k, betas=betas, eps=eps, weight_decay=weight_decay)
        super(Ranger, self).__init__(params, defaults)
        self._step = 0

        for group in self.param_groups:
            group["slow_params"] = [p.clone().detach() for p in group["params"] if p.requires_grad]

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p, sp in zip(group["params"], group["slow_params"]):
                if p.grad is None:
                    continue

                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError("Ranger does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p.data)
                    state["exp_avg_sq"] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                beta1, beta2 = group["betas"]

                state["step"] += 1
                self._step += 1

                # Apply weight decay
                if group["weight_decay"] != 0:
                    grad = grad.add(p.data, alpha=group["weight_decay"])

                # Update exponential moving averages
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Compute rectified term (RAdam)
                bias_correction1 = 1 - beta1 ** state["step"]
                bias_correction2 = 1 - beta2 ** state["step"]
                n_sma_max = 2 / (1 - beta2) - 1
                n_sma = n_sma_max - 2 * state["step"] * (beta2 ** state["step"]) / bias_correction2

                if n_sma >= 5:
                    step_size = group["lr"] * math.sqrt(
                        ((1 - beta2 ** state["step"]) * (n_sma - 4) / (n_sma_max - 4)) *
                        ((n_sma - 2) / n_sma) * (n_sma_max / (n_sma_max - 2))
                    ) / bias_correction1
                    denom = exp_avg_sq.sqrt().add_(group["eps"])
                    p.data.addcdiv_(exp_avg, denom, value=-step_size)
                else:
                    step_size = group["lr"] / bias_correction1
                    p.data.add_(exp_avg, alpha=-step_size)

                # Lookahead updates
                if self._step % group["k"] == 0:
                    sp.add_(p.data - sp, alpha=group["alpha"])
                    p.data.copy_(sp)

        return loss


In [45]:
def recurrent_summary(model, input_size):
    """
    Custom summary function that emulates torchinfo's output while correctly
    counting parameters for RNN/GRU/LSTM layers.

    This function is designed for models whose direct children are
    nn.Linear, nn.RNN, nn.GRU, or nn.LSTM layers.

    Args:
        model (nn.Module): The model to analyze.
        input_size (tuple): Shape of the input tensor (e.g., (seq_len, features)).
    """

    # Dictionary to store output shapes captured by forward hooks
    output_shapes = {}
    # List to track hook handles for later removal
    hooks = []

    def get_hook(name):
        """Factory function to create a forward hook for a specific module."""
        def hook(module, input, output):
            # Handle RNN layer outputs (returns a tuple)
            if isinstance(output, tuple):
                # output[0]: all hidden states with shape (batch, seq_len, hidden*directions)
                shape1 = list(output[0].shape)
                shape1[0] = -1  # Replace batch dimension with -1

                # output[1]: final hidden state h_n (or tuple (h_n, c_n) for LSTM)
                if isinstance(output[1], tuple):  # LSTM case: (h_n, c_n)
                    shape2 = list(output[1][0].shape)  # Extract h_n only
                else:  # RNN/GRU case: h_n only
                    shape2 = list(output[1].shape)

                # Replace batch dimension (middle position) with -1
                shape2[1] = -1

                output_shapes[name] = f"[{shape1}, {shape2}]"

            # Handle standard layer outputs (e.g., Linear)
            else:
                shape = list(output.shape)
                shape[0] = -1  # Replace batch dimension with -1
                output_shapes[name] = f"{shape}"
        return hook

    # 1. Determine the device where model parameters reside
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")  # Fallback for models without parameters

    # 2. Create a dummy input tensor with batch_size=1
    dummy_input = torch.randn(1, *input_size).to(device)

    # 3. Register forward hooks on target layers
    # Iterate through direct children of the model (e.g., self.rnn, self.classifier)
    for name, module in model.named_children():
        if isinstance(module, (nn.Linear, nn.RNN, nn.GRU, nn.LSTM)):
            # Register the hook and store its handle for cleanup
            hook_handle = module.register_forward_hook(get_hook(name))
            hooks.append(hook_handle)

    # 4. Execute a dummy forward pass in evaluation mode
    model.eval()
    with torch.no_grad():
        try:
            model(dummy_input)
        except Exception as e:
            print(f"Error during dummy forward pass: {e}")
            # Clean up hooks even if an error occurs
            for h in hooks:
                h.remove()
            return

    # 5. Remove all registered hooks
    for h in hooks:
        h.remove()

    # --- 6. Print the summary table ---

    print("-" * 79)
    # Column headers
    print(f"{'Layer (type)':<25} {'Output Shape':<28} {'Param #':<18}")
    print("=" * 79)

    total_params = 0
    total_trainable_params = 0

    # Iterate through modules again to collect and display parameter information
    for name, module in model.named_children():
        if name in output_shapes:
            # Count total and trainable parameters for this module
            module_params = sum(p.numel() for p in module.parameters())
            trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

            total_params += module_params
            total_trainable_params += trainable_params

            # Format strings for display
            layer_name = f"{name} ({type(module).__name__})"
            output_shape_str = str(output_shapes[name])
            params_str = f"{trainable_params:,}"

            print(f"{layer_name:<25} {output_shape_str:<28} {params_str:<15}")

    print("=" * 79)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {total_trainable_params:,}")
    print(f"Non-trainable params: {total_params - total_trainable_params:,}")
    print("-" * 79)

In [46]:
class CSHN_ConvBlock(nn.Module):
    """ Implements the CSHN Core: Conv1D -> BatchNorm1D -> LeakyReLU. """
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super(CSHN_ConvBlock, self).__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.LeakyReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class CSHN_FeatureExtractor(nn.Module):
    """ 3-layer CNN feature extractor with tunable filter sizes. """
    def __init__(self, num_raw_features, c1_filters, c2_filters, c3_filters, cnn_dropout_rate):
        super(CSHN_FeatureExtractor, self).__init__()
        
        # Fixed Kernel/Padding based on your architecture
        C1_KERNEL, C2_KERNEL, C3_KERNEL = 8, 5, 3 
        C2_PADDING, C3_PADDING = 2, 1
        
        # C1 (Input features -> C1_FILTERS)
        self.c1 = CSHN_ConvBlock(num_raw_features, c1_filters, C1_KERNEL, padding=0)
        # C2 (C1_FILTERS -> C2_FILTERS)
        self.c2 = CSHN_ConvBlock(c1_filters, c2_filters, C2_KERNEL, padding=C2_PADDING)  
        # C3 (C2_FILTERS -> C3_FILTERS)
        self.c3 = CSHN_ConvBlock(c2_filters, c3_filters, C3_KERNEL, padding=C3_PADDING)
        
        self.dropout = nn.Dropout(cnn_dropout_rate)

    def forward(self, x):
        # x enters as (Batch, Time, Features) -> Permute to (Batch, Features, Time) for Conv1D
        x = x.transpose(1, 2)
        
        x = self.c1(x) 
        x = self.c2(x) 
        x = self.c3(x) 
        
        # Permute back for the RNN input: (Batch, Features, Time) -> (Batch, Time, Features)
        x = x.transpose(1, 2)
        
        x = self.dropout(x)
        return x

In [47]:
class RecurrentClassifier(nn.Module):
    """
    Generic RNN classifier (RNN, LSTM, GRU).
    Uses the last hidden state for classification.
    """
    def __init__(
            self,
            input_size,
            hidden_size,
            num_layers,
            num_classes,
            rnn_type='GRU',        # 'RNN', 'LSTM', or 'GRU'
            bidirectional=False,
            dropout_rate=0.2
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        # Map string name to PyTorch RNN class
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]

        # Dropout is only applied between layers (if num_layers > 1)
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Create the recurrent layer
        self.rnn = rnn_module(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # Input shape: (batch, seq_len, features)
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Final classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    def forward(self, x):
        """
        x shape: (batch_size, seq_length, input_size)
        """

        # rnn_out shape: (batch_size, seq_len, hidden_size * num_directions)
        rnn_out, hidden = self.rnn(x)

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        # hidden shape: (num_layers * num_directions, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        return logits

In [48]:
class CSHN_HybridClassifier(nn.Module):
    """ Combines the CNN feature extractor and the RNN classification head. """
    def __init__(self, cnn_params: dict, rnn_params: dict, num_raw_features: int, num_classes: int):
        super(CSHN_HybridClassifier, self).__init__()
        
        # 1. Feature Extractor (CNN)
        self.cnn = CSHN_FeatureExtractor(
            num_raw_features=num_raw_features,
            c1_filters=cnn_params['c1_filters'],
            c2_filters=cnn_params['c2_filters'],
            c3_filters=cnn_params['c3_filters'],
            cnn_dropout_rate=cnn_params['cnn_dropout']
        )
        
        # The input size for the RNN is the output channel count of the final CNN layer (C3_FILTERS)
        rnn_input_size = cnn_params['c3_filters']
        
        # 2. Recurrent Classification Head (RNN)
        self.rnn = RecurrentClassifier(
            input_size=rnn_input_size,
            hidden_size=rnn_params['hidden_size'],
            num_layers=rnn_params['num_layers'],
            num_classes=num_classes,
            dropout_rate=rnn_params['rnn_dropout'],
            bidirectional=rnn_params['bidirectional'],
            rnn_type=rnn_params['rnn_type']
        )
        
    def forward(self, x):
        # x shape: (N, T_in, F_in)
        x = self.cnn(x)
        # x shape after CNN: (N, T_out, F_out) - This is the sequence for the RNN
        x = self.rnn(x)
        # x shape after RNN: (N, num_classes)
        return x

In [49]:
def initialize_weights(model: nn.Module, init_scheme: str):
    """
    Initializes weights, correctly skipping 1D tensors (like biases) 
    that cause 'Fan in and fan out' errors.
    """
    for name, param in model.named_parameters():
        # Skip parameters with fewer than 2 dimensions (typically biases and BatchNorm params)
        if param.dim() < 2:
            # Optionally initialize biases to zero, or skip them
            if 'bias' in name:
                torch.nn.init.constant_(param.data, 0.0)
            continue

        # Apply initialization schemes only to weight matrices (dim >= 2)
        if init_scheme == 'xavier_uniform':
            torch.nn.init.xavier_uniform_(param.data)
        elif init_scheme == 'kaiming_normal':
            # Good for ReLU/Leaky ReLU (common in surrounding layers)
            torch.nn.init.kaiming_normal_(param.data, mode='fan_in', nonlinearity='leaky_relu')
        elif init_scheme == 'orthogonal':
            torch.nn.init.orthogonal_(param.data)


In [50]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [51]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [52]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [53]:
import torch
import torch.nn as nn
import optuna
# Importa la funzione di utilità se necessario (anche se lo faremo manualmente) 7
# import torch.nn.utils as nn_utils 

def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name="",
        trial=None, scheduler=None, weight_max_norm=None):
    """
    Train the neural network model on the training data and validate on the validation data.
    """

    # --- NUOVA FUNZIONE: Normalizzazione/Vincolo dei Pesi ---
    def apply_weight_constraint(model, max_norm):
        with torch.no_grad():
            for name, module in model.named_modules():
                # Applica solo ai layer con pesi (es. Linear, Conv1d, Conv2d)
                if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
                    # Vincola il parametro 'weight'
                    if hasattr(module, 'weight') and module.weight is not None:
                        # Calcola la norma L2 del tensore dei pesi
                        norm = module.weight.norm(2)
                        
                        if norm > max_norm:
                            # Se la norma è maggiore del vincolo, riscala il peso.
                            # Ciò garantisce che la norma L2 sia esattamente 'max_norm' o inferiore.
                            module.weight.data.mul_(max_norm / norm)

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Initialize best_metric before loop
    best_metric = float('-inf') if mode == 'max' else float('inf')
    best_epoch = 0
    
    if patience > 0:
        patience_counter = 0

    if verbose > 0: # Print only if verbose
        print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
        
        if weight_max_norm is not None and weight_max_norm > 0:
            apply_weight_constraint(model, weight_max_norm)

        # Evaluate model on validation data
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write to TensorBoard
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                      f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                      f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Get current metric for Pruning & Early Stopping
        current_metric = training_history[evaluation_metric][-1]

        # Scheduler Step
        if scheduler is not None:
            scheduler.step(current_metric)
        
        # Optuna Pruning
        if trial is not None:
            try:
                trial.report(current_metric, epoch)
            except Exception as e:
                # Gestione dell'errore (solo se Optuna è effettivamente usato)
                print(f"Error during Optuna report: {e}") 
                pass

            if trial.should_prune():
                # Pruning requested
                raise optuna.exceptions.TrialPruned()
        # End Pruning

        # Early stopping logic (omesso per brevità, resta invariato)
        if patience > 0:
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                # Non uso la variabile `experiment_name` qui, assumo che sia definita in un contesto più ampio
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if verbose > 0:
                        print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights (omesso per brevità, resta invariato)
    if restore_best_weights and patience > 0 and best_epoch > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        if verbose > 0:
            print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping (omesso per brevità, resta invariato)
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
        if mode == 'max':
            best_metric = max(training_history[evaluation_metric])
        else:
            best_metric = min(training_history[evaluation_metric])
            
    if patience > 0 and best_epoch == 0:
         if mode == 'max':
             best_metric = max(training_history[evaluation_metric])
         else:
             best_metric = min(training_history[evaluation_metric])


    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history, best_metric

## Optuna

In [54]:
from sklearn.utils.class_weight import compute_class_weight
BATCH_SIZE = 32
EPOCHS = 200
PATIENCE = 15 

def set_seed(seed_value):
    """Set seeds for reproducibility across different components."""
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # for multi-GPU

def objective(trial):

    seed_hp = trial.suggest_int("SEED", 0, 10000) 
    set_seed(seed_hp) 

    sid_labels = df_train_fold[['sample_index', 'label_encoded']].drop_duplicates()
    
    sids_to_split = sid_labels['sample_index'].values
    sid_stratify_labels = sid_labels['label_encoded'].values
    
    train_users, val_users = train_test_split(
        sids_to_split,
        test_size=0.2,
        stratify=sid_stratify_labels,
        random_state=seed_hp
    )

    
    window_hp = trial.suggest_categorical("WINDOW", [20, 40])
    stride_hp = trial.suggest_categorical("STRIDE", [5, 10, 15, 20, 30])

    if stride_hp >= window_hp:
        raise optuna.exceptions.TrialPruned()
    
    # --- 2. Build Dynamic Datasets and Loaders ---
    # Generate sequences for this trial
    X_train, y_train = build_sequences(
        df_train_fold, 
        feature_cols=feature_cols, 
        id_col='sample_index',
        label_col='label_encoded',
        window=window_hp, 
        stride=stride_hp
    )
    X_val, y_val = build_sequences(
        df_val_fold, 
        feature_cols=feature_cols,
        id_col='sample_index',
        label_col='label_encoded',
        window=window_hp, 
        stride=stride_hp
    )
    
    labels = np.unique(y_train)

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=labels,
        y=y_train
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
   
    
    ## Reshaping of the data for the CNN


    X_train_reshaped = X_train.reshape(-1, num_features) 

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled_reshaped = scaler.fit_transform(X_train_reshaped)

    X_train_scaled = X_train_scaled_reshaped.reshape(
        X_train.shape[0],
        window_hp, 
        num_features
    )

    X_val_reshaped = X_val.reshape(-1, num_features) 
    
    # Use the scaler FITTED on the training data
    X_val_scaled_reshaped = scaler.transform(X_val_reshaped)

    # Reshape back to 3D for the model: (N_samples, Time, Features)
    X_val_scaled = X_val_scaled_reshaped.reshape(
        X_val.shape[0],
        window_hp, 
        num_features
    )
    
    # Create TensorDatasets for this trial
    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_train_scaled), torch.from_numpy(y_train))
    val_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_val_scaled), torch.from_numpy(y_val))

    # Create DataLoaders for this trial
    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    
    # --- 3. Suggest Other Hyperparameters ---
    
    c1_filters_hp = trial.suggest_categorical("c1_filters", [16, 32])
    
    c2_filters_hp = trial.suggest_categorical("c2_filters", [32, 48, 64, 96])
    
    if c1_filters_hp == 16 and c2_filters_hp not in [32, 48]:
         raise optuna.exceptions.TrialPruned()
    if c1_filters_hp == 32 and c2_filters_hp not in [64, 96]:
         raise optuna.exceptions.TrialPruned()

    c3_filters_hp = trial.suggest_categorical("c3_filters", [64, 96, 128, 144, 192, 288])
    
    if c3_filters_hp not in [c2_filters_hp * 2, c2_filters_hp * 3]:
        raise optuna.exceptions.TrialPruned()

    cnn_dropout_hp = trial.suggest_float("cnn_dropout", 0.1, 0.4) 
    
    
    # --- 4. Suggest RNN/Classification Head Hyperparameters ---
    
    hidden_size_hp = trial.suggest_categorical("hidden_size", [32, 64, 128])
    num_layers_hp = trial.suggest_int("num_layers", 1, 3)
    rnn_dropout_hp = trial.suggest_float("rnn_dropout", 0.1, 0.7) 
    rnn_type_hp = trial.suggest_categorical("rnn_type", ["GRU", "LSTM"]) 
    bidirectional_hp = trial.suggest_categorical("bidirectional", [True, False])
    
    init_scheme_hp = trial.suggest_categorical(
        "init_scheme", ["xavier_uniform", "orthogonal", "kaiming_normal"]
    )
    
    # Optimization & Loss Parameters
    lr_hp = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay_hp = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    l1_lambda_hp = trial.suggest_float("l1_lambda", 1e-7, 1e-4, log=True)
    focal_gamma_hp = trial.suggest_float("focal_gamma", 0.0, 5.0, step=0.5)
    weight_max_norm_hp = trial.suggest_float("weight_max_norm", low=0.1, high=6.0, log=False)
    
    # Scheduler Parameters
    scheduler_patience_hp = trial.suggest_int("scheduler_patience", 3, 10, step=1)
    scheduler_factor_hp = trial.suggest_categorical("scheduler_factor", [0.1, 0.2, 0.5])

    # --- 5. Create Model, Criterion, Optimizer, and Scheduler ---
    
    cnn_params = {
        'c1_filters': c1_filters_hp,
        'c2_filters': c2_filters_hp,
        'c3_filters': c3_filters_hp,
        'cnn_dropout': cnn_dropout_hp
    }
    
    rnn_params = {
        'hidden_size': hidden_size_hp,
        'num_layers': num_layers_hp,
        'rnn_dropout': rnn_dropout_hp,
        'bidirectional': bidirectional_hp,
        'rnn_type': rnn_type_hp
    }
    
    # Model Definition: Uses the new Hybrid Classifier
    model = CSHN_HybridClassifier(
        cnn_params=cnn_params,
        rnn_params=rnn_params,
        num_raw_features=len(feature_cols),
        num_classes=3
    ).to(device)

    # Criterion: Using FocalLoss
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    
    # Apply Initialization
    initialize_weights(model, init_scheme=init_scheme_hp) 

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_hp, weight_decay=weight_decay_hp)
    
    # Scheduler Definition (ReduceLROnPlateau)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=scheduler_factor_hp, patience=scheduler_patience_hp, 
        verbose=False, threshold=1e-4, min_lr=1e-7
    )

    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    
    # --- 6. Run Training (The 'fit' function remains the same!) ---
    os.makedirs('models', exist_ok=True)
    try:
        _, _, best_val_f1 = fit(
            model=model, train_loader=train_loader, val_loader=val_loader, epochs=EPOCHS,
            criterion=criterion, optimizer=optimizer, scaler=scaler, device=device,
            l1_lambda=l1_lambda_hp, l2_lambda=0, patience=PATIENCE, evaluation_metric="val_f1",
            mode='max', restore_best_weights=True, trial=trial, scheduler=scheduler, 
            weight_max_norm=weight_max_norm_hp, writer=None, verbose=1, experiment_name=f"optuna_trial_{trial.number}"
        )
        
        return best_val_f1

    except optuna.exceptions.TrialPruned:
        return 0.0 
    except Exception as e:
        print(f"Trial {trial.number} failed with exception: {e}")
        return 0.0

# --- 6. Create and Run the Optuna Study ---
print("--- Starting Optuna Hyperparameter Tuning ---")

study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_startup_trials=3)
)

study.optimize(objective, n_trials=50) 

print("\n--- Tuning Complete ---")
print(f"Best trial number: {study.best_trial.number}")
print(f"Best validation F1-score: {study.best_value:.4f}")
print("Best hyperparameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2025-11-15 11:09:39,729] A new study created in memory with name: no-name-46baf9da-c774-4477-a8d3-217ee2c1f721


--- Starting Optuna Hyperparameter Tuning ---


[I 2025-11-15 11:09:40,300] Trial 0 pruned. 
[I 2025-11-15 11:09:40,896] Trial 1 pruned. 
[I 2025-11-15 11:09:41,448] Trial 2 pruned. 
[I 2025-11-15 11:09:42,037] Trial 3 pruned. 


Training 200 epochs...
Epoch   1/200 | Train: Loss=1.2452, F1 Score=0.6532 | Val: Loss=1.0895, F1 Score=0.0357
Epoch   2/200 | Train: Loss=0.9562, F1 Score=0.8174 | Val: Loss=0.6039, F1 Score=0.8412
Epoch   3/200 | Train: Loss=0.7264, F1 Score=0.8713 | Val: Loss=0.4345, F1 Score=0.8826
Epoch   4/200 | Train: Loss=0.6048, F1 Score=0.8919 | Val: Loss=0.3073, F1 Score=0.9278
Epoch   5/200 | Train: Loss=0.5162, F1 Score=0.9030 | Val: Loss=0.2844, F1 Score=0.9140
Epoch   6/200 | Train: Loss=0.4605, F1 Score=0.9140 | Val: Loss=0.2868, F1 Score=0.9033
Epoch   7/200 | Train: Loss=0.4310, F1 Score=0.9189 | Val: Loss=0.3149, F1 Score=0.8955
Epoch   8/200 | Train: Loss=0.3925, F1 Score=0.9278 | Val: Loss=0.2458, F1 Score=0.9197
Epoch   9/200 | Train: Loss=0.3779, F1 Score=0.9265 | Val: Loss=0.2061, F1 Score=0.9389
Epoch  10/200 | Train: Loss=0.3495, F1 Score=0.9373 | Val: Loss=0.2016, F1 Score=0.9335
Epoch  11/200 | Train: Loss=0.3374, F1 Score=0.9358 | Val: Loss=0.2842, F1 Score=0.8937
Epoch  12

[I 2025-11-15 11:12:46,768] Trial 4 finished with value: 0.9577584119036775 and parameters: {'SEED': 2325, 'WINDOW': 20, 'STRIDE': 5, 'c1_filters': 16, 'c2_filters': 32, 'c3_filters': 96, 'cnn_dropout': 0.2734923240298178, 'hidden_size': 128, 'num_layers': 1, 'rnn_dropout': 0.3138838861767397, 'rnn_type': 'GRU', 'bidirectional': False, 'init_scheme': 'xavier_uniform', 'lr': 2.169532471604797e-05, 'weight_decay': 5.035174288856775e-06, 'l1_lambda': 3.9774350145913914e-05, 'focal_gamma': 3.0, 'weight_max_norm': 0.5887931037379235, 'scheduler_patience': 9, 'scheduler_factor': 0.1}. Best is trial 4 with value: 0.9577584119036775.
[I 2025-11-15 11:12:46,782] Trial 5 pruned. 


...

Training 200 epochs...
Epoch   1/200 | Train: Loss=0.8651, F1 Score=0.7967 | Val: Loss=0.6786, F1 Score=0.7591
Epoch   2/200 | Train: Loss=0.5513, F1 Score=0.8900 | Val: Loss=0.3333, F1 Score=0.9111
Epoch   3/200 | Train: Loss=0.3688, F1 Score=0.9115 | Val: Loss=0.3214, F1 Score=0.9073
Epoch   4/200 | Train: Loss=0.2794, F1 Score=0.9225 | Val: Loss=0.2974, F1 Score=0.9007
Epoch   5/200 | Train: Loss=0.2135, F1 Score=0.9305 | Val: Loss=0.2943, F1 Score=0.9306
Epoch   6/200 | Train: Loss=0.1775, F1 Score=0.9381 | Val: Loss=0.2567, F1 Score=0.9308
Epoch   7/200 | Train: Loss=0.1522, F1 Score=0.9462 | Val: Loss=0.2317, F1 Score=0.9541
Epoch   8/200 | Train: Loss=0.1438, F1 Score=0.9499 | Val: Loss=0.3076, F1 Score=0.9387
Epoch   9/200 | Train: Loss=0.1388, F1 Score=0.9526 | Val: Loss=0.3029, F1 Score=0.9148
Epoch  10/200 | Train: Loss=0.1094, F1 Score=0.9600 | Val: Loss=0.2412, F1 Score=0.9533
Epoch  11/200 | Train: Loss=0.1029, F1 Score=0.9650 | Val: Loss=0.3049, F1 Score=0.9428
Epoch  12

[I 2025-11-15 11:35:31,400] Trial 48 finished with value: 0.9541094516347863 and parameters: {'SEED': 3120, 'WINDOW': 40, 'STRIDE': 5, 'c1_filters': 16, 'c2_filters': 32, 'c3_filters': 96, 'cnn_dropout': 0.3450307543265989, 'hidden_size': 32, 'num_layers': 3, 'rnn_dropout': 0.22796351649616273, 'rnn_type': 'LSTM', 'bidirectional': False, 'init_scheme': 'xavier_uniform', 'lr': 0.00012186163695640763, 'weight_decay': 1.9782652869579814e-06, 'l1_lambda': 3.607647679872166e-06, 'focal_gamma': 2.0, 'weight_max_norm': 3.732005552873669, 'scheduler_patience': 9, 'scheduler_factor': 0.2}. Best is trial 4 with value: 0.9577584119036775.


Epoch  22/200 | Train: Loss=0.0586, F1 Score=0.9804 | Val: Loss=0.3103, F1 Score=0.9480
Early stopping triggered after 22 epochs.
Best model restored from epoch 7 with val_f1 0.9541
Training 200 epochs...
Epoch   1/200 | Train: Loss=0.7172, F1 Score=0.8171 | Val: Loss=0.9109, F1 Score=0.6858
Epoch   2/200 | Train: Loss=0.3920, F1 Score=0.8939 | Val: Loss=0.3401, F1 Score=0.8835
Epoch   3/200 | Train: Loss=0.2896, F1 Score=0.9153 | Val: Loss=0.2583, F1 Score=0.9272
Epoch   4/200 | Train: Loss=0.2387, F1 Score=0.9269 | Val: Loss=0.2339, F1 Score=0.9349
Epoch   5/200 | Train: Loss=0.1969, F1 Score=0.9372 | Val: Loss=0.2957, F1 Score=0.9201
Epoch   6/200 | Train: Loss=0.1831, F1 Score=0.9419 | Val: Loss=0.2342, F1 Score=0.9315
Epoch   7/200 | Train: Loss=0.1621, F1 Score=0.9507 | Val: Loss=0.3028, F1 Score=0.9171
Epoch   8/200 | Train: Loss=0.1452, F1 Score=0.9532 | Val: Loss=0.3053, F1 Score=0.9272
Epoch   9/200 | Train: Loss=0.1447, F1 Score=0.9495 | Val: Loss=0.2038, F1 Score=0.9473
Epo

[I 2025-11-15 11:38:06,651] Trial 49 finished with value: 0.9525633142628056 and parameters: {'SEED': 9435, 'WINDOW': 40, 'STRIDE': 5, 'c1_filters': 16, 'c2_filters': 32, 'c3_filters': 96, 'cnn_dropout': 0.32837074389990406, 'hidden_size': 128, 'num_layers': 3, 'rnn_dropout': 0.10984320482501711, 'rnn_type': 'LSTM', 'bidirectional': False, 'init_scheme': 'orthogonal', 'lr': 0.00010916137191969961, 'weight_decay': 2.010534537272591e-06, 'l1_lambda': 2.103243728062754e-06, 'focal_gamma': 2.0, 'weight_max_norm': 3.1547088062068154, 'scheduler_patience': 9, 'scheduler_factor': 0.1}. Best is trial 4 with value: 0.9577584119036775.


Epoch  25/200 | Train: Loss=0.0616, F1 Score=0.9803 | Val: Loss=0.3558, F1 Score=0.9386
Early stopping triggered after 25 epochs.
Best model restored from epoch 10 with val_f1 0.9526

--- Tuning Complete ---
Best trial number: 4
Best validation F1-score: 0.9578
Best hyperparameters found:
  SEED: 2325
  WINDOW: 20
  STRIDE: 5
  c1_filters: 16
  c2_filters: 32
  c3_filters: 96
  cnn_dropout: 0.2734923240298178
  hidden_size: 128
  num_layers: 1
  rnn_dropout: 0.3138838861767397
  rnn_type: GRU
  bidirectional: False
  init_scheme: xavier_uniform
  lr: 2.169532471604797e-05
  weight_decay: 5.035174288856775e-06
  l1_lambda: 3.9774350145913914e-05
  focal_gamma: 3.0
  weight_max_norm: 0.5887931037379235
  scheduler_patience: 9
  scheduler_factor: 0.1


# Visulization and evaluation of the optuna study

In [61]:
def load_model_from_study(study: optuna.study.Study, trial_number: int, num_classes: int, device: torch.device):
    """
    Placeholder: Instantiates the CSHN_HybridClassifier model using the
    hyperparameters from a specific Optuna trial and loads saved weights.
    """
    trial = next(t for t in study.trials if t.number == trial_number)
    params = trial.params
        
        # 1. Extract and Validate Params
    cnn_params = {
        'c1_filters': params['c1_filters'],
        'c2_filters': params['c2_filters'],
        'c3_filters': params['c3_filters'],
        'cnn_dropout': params['cnn_dropout']
    }
    rnn_params = {
        'hidden_size': params['hidden_size'],
        'num_layers': params['num_layers'],
        'rnn_dropout': params['rnn_dropout'],
        'bidirectional': params['bidirectional'],
        'rnn_type': params['rnn_type']
    }
        
        # 2. Instantiate Model
    model = CSHN_HybridClassifier(
        cnn_params=cnn_params,
        rnn_params=rnn_params,
        num_raw_features=len(feature_cols),
        num_classes=num_classes
    ).to(device)

    model_path = f"models/optuna_trial_{trial_number}_model.pt"
    try:
        model.load_state_dict(torch.load(model_path))
        print(f"Successfully loaded model from trial {trial_number} -> {model_path}")
        return model
    except FileNotFoundError:
        print(f"ERROR: Could not find model file {model_path}.")
        print("This might happen if the trial was pruned before saving a model.")
        return None

In [56]:
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
    plot_intermediate_values
)

## Predict Public tests

In [62]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# --- HELPER FUNCTION: CREATE SLIDING WINDOWS ---
def create_sliding_windows(X_df_full, feature_cols, seq_length, step):
    """Generates sliding windows/sequences for test/inference data."""
    X_sequences, sample_index_map = [], []
    grouped = X_df_full.groupby('sample_index')
    for sample_id, user_data in grouped:
        user_features = user_data[feature_cols].values.astype(np.float32)
        # Iterate over the time series to create overlapping windows
        for i in range(0, len(user_features) - seq_length + 1, step):
            X_sequences.append(user_features[i:i + seq_length])
            sample_index_map.append(sample_id)
    return np.array(X_sequences, dtype=np.float32), sample_index_map

In [63]:
from typing import Tuple, List, Any, Callable, Optional 
from datetime import datetime

def generate_submission_from_trial(
    study: optuna.study.Study,
    trial_number: int,
    num_classes: int,
    device: torch.device,
    df_test: pd.DataFrame,
    feature_cols: List[str],
    BATCH_SIZE: int,
    make_loader: Callable,
    inverse_label_map: dict
) -> Optional[pd.DataFrame]:
    """
    Generalized test inference pipeline for a given Optuna trial.

    Args:
        study (optuna.Study): Your Optuna study object.
        trial_number (int): Trial number to load.
        num_classes (int): Number of output classes.
        device (torch.device): CPU or GPU.
        df_test (pd.DataFrame): Test dataframe with 'sample_index' and features.
        feature_cols (list): List of feature columns.
        BATCH_SIZE (int): Batch size for inference.
        make_loader (Callable): Function to create DataLoader.
        inverse_label_map (dict): Map numeric labels to string labels.

    Returns:
        submission_df (pd.DataFrame): Submission with 'sample_index' and 'label', or None on failure.
    """

    # --- 1. Load the model ---
    model = load_model_from_study(
        study=study,
        trial_number=trial_number,
        num_classes=num_classes,
        device=device
    )
    if model is None:
        return None

    # --- 2. Get WINDOW and STRIDE from trial ---
    try:
        trial = study.trials[trial_number]
    except IndexError:
        # Fallback if trial_number is not index but actual trial.number
        trial = next(t for t in study.trials if t.number == trial_number)
        
    SEQ_LENGTH = trial.params["WINDOW"]
    STEP = trial.params["STRIDE"]
    print(f"Inference using WINDOW={SEQ_LENGTH}, STRIDE={STEP}")

    X_test_seq, test_index_map = create_sliding_windows(
        df_test, feature_cols, SEQ_LENGTH, STEP
    )

    print(f"Created test sequences: {X_test_seq.shape}")
    # --- 4. TensorDataset & DataLoader ---
    X_test_tensor = torch.from_numpy(X_test_seq)
    test_ds = TensorDataset(X_test_tensor)
    test_loader = make_loader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    # --- 5. Generate predictions ---
    model.eval()
    all_logits = []
    with torch.no_grad():
        for (inputs,) in test_loader:
            inputs = inputs.to(device)
            # Use autocast only if CUDA is available, otherwise it's overhead
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
            all_logits.append(logits.cpu().numpy())

    final_logits_all_windows = np.concatenate(all_logits)
    print(f"Generated logits for {len(final_logits_all_windows)} windows.")

    # --- 6. Aggregate predictions (Majority/Average Voting) ---
    pred_df = pd.DataFrame({'sample_index': test_index_map})
    for c in range(num_classes):
        pred_df[f'logit_{c}'] = final_logits_all_windows[:, c]

    # Aggregate by averaging logits and finding the maximum averaged logit
    submission_logits_avg = pred_df.groupby('sample_index')[[f'logit_{c}' for c in range(num_classes)]].mean()
    final_numeric_predictions = submission_logits_avg.idxmax(axis=1).str.replace('logit_', '').astype(int)
    final_numeric_predictions = final_numeric_predictions.reset_index(name='prediction')
    
    # Map back to final labels
    final_labels = final_numeric_predictions['prediction'].map(inverse_label_map)

    submission_df = pd.DataFrame({
        'sample_index': final_numeric_predictions['sample_index'],
        'label': final_labels
    })

    # --- 7. Save Submission ---
    filename = f'trial_{trial_number}_{datetime.now().strftime("%Y%m%d_%H%M%S")}_submission.csv'
    submission_df.to_csv(filename, index=False)
    print(f"\n✅ Submission saved to: {filename} with {len(submission_df)} rows.")
    print(submission_df.head(50))

    return submission_df
    
trial_num = 4
inverse_map = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}  

submission_df = generate_submission_from_trial(
        study=study,
        trial_number=trial_num,
        num_classes=3,
        device=device,
        df_test=df_public_test,
        feature_cols=feature_cols,
        BATCH_SIZE=BATCH_SIZE,
        make_loader=make_loader,
        inverse_label_map=inverse_map
)


Successfully loaded model from trial 4 -> models/optuna_trial_4_model.pt
Inference using WINDOW=20, STRIDE=5
Created test sequences: (38396, 20, 32)
Generated logits for 38396 windows.

✅ Submission saved to: trial_4_20251115_120925_submission.csv with 1324 rows.
    sample_index      label
0              0   low_pain
1              1    no_pain
2              2    no_pain
3              3    no_pain
4              4    no_pain
5              5    no_pain
6              6  high_pain
7              7    no_pain
8              8    no_pain
9              9    no_pain
10            10    no_pain
11            11    no_pain
12            12    no_pain
13            13    no_pain
14            14    no_pain
15            15    no_pain
16            16   low_pain
17            17    no_pain
18            18  high_pain
19            19    no_pain
20            20    no_pain
21            21    no_pain
22            22  high_pain
23            23   low_pain
24            24  high_pain
25      